In [2]:
import re

import cv2
import pytesseract
from PIL import Image
import pandas as pd
import json

In [3]:
# Si estás en Windows, quizá necesites especificar el path manualmente:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Abrir imagen
img = cv2.imread("IMG_3472.jpg")

# Extraer texto (en español)
text = pytesseract.image_to_string(img, lang="spa")

In [4]:
lines = text.split('\n')

In [5]:
CODE_RE = r"[0-9/]{13}"
QP_RE = r"(\d,\d{4}) u [Xx] (\d{3,5},\d{3,4})"

codes = list(); prices = list(); quantities = list()

def re_findall(expression, line):
    findings =re.findall(expression, line)
    if findings: return findings

def float_conversion(float_text):
    return float(float_text.replace(',','.'))

for line in lines:
    
    # Code
    if code := re_findall(CODE_RE, line):
        codes.extend(code)

    # Quantity
    if qp := re_findall(QP_RE,line):
        quantity, price = qp[0]
        quantities.append(float_conversion(quantity)); prices.append(float_conversion(price))

products = zip(codes, prices, quantities)

data = list(products)
df = pd.DataFrame(data, columns=['Code', 'Price', 'Units'])
df['Payment'] = round(df['Price'] * df['Units'], 2)
df.set_index('Code', inplace=True)

In [6]:
df_agg = df.groupby(['Code', 'Price'])['Units'].sum()

In [7]:
df_agg = df_agg.reset_index()

In [8]:
df_agg

,Code,Price,Units
0,1750040136243,3216.79,1.000
1,2220940002829,36416.69,0.282
2,7/91813423386,2699.99,1.000
3,7790580131364,4949.69,1.000
4,7790748235095,2599.99,1.000
5,7791290792043,4799.99,1.000
6,7791290796058,4369.99,1.000
7,7791828000077,3499.99,2.000
8,7793560000089,1145.99,1.000
9,7793654000023,540.00,1.000


In [9]:
df_agg.set_index('Code', inplace=True)

In [15]:
df_agg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 1750040136243 to 7793654000023
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Price   10 non-null     float64
 1   Units   10 non-null     float64
dtypes: float64(2)
memory usage: 540.0+ bytes


In [16]:
df_agg.to_json(r'C:\Users\nical\OneDrive\Coding\Super\data\receipts\new_receipt.json', indent=4, orient='index')

In [13]:
df['Payment'].sum()

41591.91999999999

In [22]:
ACTUAL_TICKET = r"C:\Users\nical\OneDrive\Coding\Super\data\receipts\new_receipt.json"

def open_json(path=ACTUAL_TICKET):
    with open(path, "r", encoding="utf-8") as json_file:
        return json.load(json_file)

actual_ticket = open_json()

In [24]:
for code in actual_ticket:
    print(code)

1750040136243
2220940002829
7791813423386
7790580131364
7790748235095
7791290792043
7791290796058
7791828000077
7793560000089
7793654000023
